# Migrating from R `mboost`

R `mboost` is the R package for model-based boosting whose interface mboostrs
follows; "upstream" below means R mboost (or R gamboostLSS for distributional
models). The tables map each common R `mboost` call to its mboostrs
equivalent.

## Workflow mapping

The table maps fitting, cross-validation, path truncation and prediction
from R `mboost` to Python. It applies directly to learners and families
without a recorded numerical deviation; the later tables identify exceptions.

| Step | R `mboost` | mboostrs |
|---|---|---|
| load | `library(mboost)` | `import mboostrs as mb` |
| fit | `gamboost(y ~ bbs(x), data = d, family = Gaussian(), control = boost_control(mstop = 300, nu = 0.1))` | `mb.gamboost("y ~ bbs(x)", data=d, family=mb.Gaussian(), control=mb.boost_control(mstop=300, nu=0.1))` |
| tune | `cvr <- cvrisk(fit, folds = cv(model.weights(fit), type = "kfold", B = 5))` | `cvr = fit.cvrisk(type="kfold", B=5)` |
| read the chosen stop | `m <- mstop(cvr)` | `m = cvr.optimal_mstop` |
| truncate | `final <- fit[m]` | `final = fit[:m]` |
| coefficients | `coef(final)` | `final.coef()` |
| predict | `predict(final, newdata = d, type = "response")` | `final.predict(newdata=d, type="response")` |
| fitted values | `fitted(final)` | `final.fitted_` |
| selected terms | `table(final$xselect())` | `final.selected_which()` |

Formulas are strings and data are supplied as a dictionary or DataFrame.
Operations that differ numerically or structurally from their upstream
counterparts are described in the notes column of the later tables, and
`fit.parity_notes()` reports the differences that apply to a fitted model.

## Import conventions

In [1]:
import mboostrs as mb
import numpy as np

| R | mboostrs |
|---|---|
| `library(mboost)` | `import mboostrs as mb` |

## Fitting

### GLMBoost (linear boosting)

| R | mboostrs |
|---|---|
| `glmboost(y ~ x1 + x2, data = d, family = Gaussian(), control = boost_control(mstop = 200, nu = 0.1))` | `mb.glmboost("y ~ x1 + x2", data=d, family=mb.Gaussian(), control=mb.boost_control(mstop=200, nu=0.1))` |

Key differences:

- Formula is a **string**, not an R formula object.
- `family = Gaussian()` becomes `family=mb.Gaussian()` (Python class instance).
- `boost_control` is `mb.boost_control` (same argument names).
- Data is a **dict of NumPy arrays** or a pandas DataFrame, not an R data.frame.

### GAMBoost (additive boosting)

| R | mboostrs |
|---|---|
| `gamboost(y ~ bbs(x1, df=4) + bols(x2), data = d, family = Gaussian())` | `mb.gamboost("y ~ bbs(x1, df=4) + bols(x2)", data=d, family=mb.Gaussian())` |

Base-learner names are the same (`bols`, `bbs`, `bns`, `bmono`, `btree`,
`brandom`, `bmrf`, `bspatial`, `brad`, `bkernel`, `buser`,
`bconstrained`). Arguments match R where possible. The table lists the
learner classes; its last column notes the four learners whose numerical
behavior differs from the upstream implementation (`btree`, `bns`, `brad`,
`bkernel`):

| R learner | mboostrs class | Key arguments |
|---|---|---|
| `bols(x, intercept=TRUE)` | `mb.Bols("x", intercept=True)` | `df`, `lambda_value`, `by` |
| `bbs(x, df=4, knots=20, degree=3, differences=2)` | `mb.Bbs("x", df=4, knots=20, degree=3, differences=2)` | `center`, `by` |
| `bmono(x, type="iterative")` | `mb.Bmono("x", type="iterative")` | `constraint`, `knots`, `degree` |
| `btree(x)` | `mb.Btree("x")` | `maxdepth`. **Splits on squared error**, not R's conditional-inference criterion, so it chooses different splits and has a different variable-selection bias |
| `brandom(g, df=4)` | `mb.Brandom("g", df=4)` | group codes may be 0-based or 1-based; see [0-based and 1-based codes](#r-migration-indexing) |
| `bmrf(x, K=K)` | `mb.Bmrf("x", K=K)` | K must be set programmatically (not in formula) |
| `bspatial(x, y, ...)` | `mb.Bspatial("x", "y", ...)` | |
| `bns(x, df=4)` (withdrawn) | `mb.Bns("x", df=4)` | **mboost 2.9-11 has removed `bns`**: calling it raises "Base-learner 'bns' has ben removed. Consider 'bbs' instead", so a script that uses it predates the removal. mboostrs still has one, **ridge-penalised** rather than the unpenalised natural spline the name implies: the fit is smoother, and `df` counts the penalised trace, not the basis dimension |
| `brad(x1, x2)` | `mb.Brad(["x1", "x2"], data)` | Matérn(3/2) as upstream, but **centres come from a farthest-point rule**, not `fields::cover.design`, so coefficients are not comparable term by term |
| `bkernel(...)` | `mb.Bkernel("x")` | The implementations fit different models: mboostrs fits a kernel ridge regression in one covariate, with the kernel evaluated between observed values of `x`; the R `bkernel` used for pathway analysis of genome-wide association data builds its kernel over sets of genetic markers. |
| `buser(X, K)` | `mb.Buser("x", X=X, K=K)` | caller-supplied design and penalty, bound through `env=` |
| `bconstrained(x, C=C)` | `mb.Bconstrained("x", C=C)` | constraint matrix bound through `env=` |

### Survival models

R uses `Surv(time, event)` in the formula. mboostrs uses the same
string-based formula:

```python
rng = np.random.default_rng(0)
n = 200
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
time = rng.exponential(size=n)
status = rng.integers(0, 2, size=n)
d = {"time": time, "status": status, "x1": x1, "x2": x2}

# R:  glmboost(Surv(time, status) ~ x1 + x2, data=d, family=CoxPH())
# mboostrs:
fit = mb.glmboost("Surv(time, status) ~ x1 + x2", data=d, family=mb.CoxPH(),
                   control=mb.boost_control(mstop=200, nu=0.1))
```

### GAMLSS boosting

| R | mboostrs |
|---|---|
| `glmboostLSS(y ~ x1 + x2, data=d, families=GaussianLSS())` | `mb.glmboostLSS("y ~ x1 + x2", data=d, family=mb.GaussianLSS())` |
| `gamboostLSS(y ~ bbs(x1), data=d, families=GaussianLSS())` | `mb.gamboostLSS("y ~ bbs(x1)", data=d, family=mb.GaussianLSS())` |
| `GaussianLSS(stabilization="MAD")` (gamboostLSS 2.2-0 family argument) | `mb.GaussianLSS(stabilization="MAD")`; the mboostrs R frontend takes the same family argument |
| a distributional fit on the response as given | `mb.gamboostLSS(..., response_scale=None)`; the default `"auto"` fits the response divided by its SD and maps the model back |
| `QuantReg(tau)` with `boost_control(nu = 0.1)` | `mb.Quantile(tau, qoffset=0.5, response_scale=None)` with `mb.boost_control(mstop, nu=0.1)`; the defaults start at the `tau`-quantile, take `nu = 0.3` when no control is given, and scale the response |
| `Laplace()` | `mb.Laplace(response_scale=None)`; the default scales the response |

`families=` is also accepted in Python for R compatibility, but `family=` is
the preferred spelling for a single LSS family. `stabilization=` (a gamboostLSS 2.2-0 family option that scales the
negative gradient before each step) is a family argument in both
mboostrs frontends, and per-parameter step lengths are
`boost_control(nu = c(mu = ..., sigma = ...))` in R, as upstream, and the
fit-level `nu=` mapping in Python.

A translated script that tunes `mstop` for R's raw-units path should pass
`response_scale=None` (Python) or `response_scale = "none"` (R) to get the
same coefficients. With the default `"auto"` the fit runs on the response
divided by its standard deviation and the model is mapped back to the
original units, so the same data in dollars and in thousands of dollars give
the same model; R gamboostLSS gives different ones, and in large units its
mean can stay at its offset for any practical `mstop`. The
[R parity and noted differences](../implementation/parity_and_bugs) chapter
gives the mechanism and the measurements.

Available LSS families, all 27: `GaussianLSS`, `GammaLSS`, `StudentTLSS`,
`BetaLSS`, `NBinomialLSS`, `NegativeBinomialLSS`, `PoissonLSS`,
`GeometricLSS`, `BinomialLSS`, `GumbelLSS`, `LogisticLSS`, `InverseGaussianLSS`, `Pareto2LSS`, `PIGLSS`,
`BCCGLSS`, `BCTLSS`, `BCPELSS`, `JSULSS`, `ZIPoLSS`, `ZINBLSS`, `ZeroAdjustedPoissonLSS`,
`ZeroAdjustedGammaLSS`, `GaussianKnownVarLSS`, `DirichletLSS`, `LogNormalLSS`, `LogLogLSS`, `WeibullLSS`. All
twenty-seven fit.

### Multinomial

| R | mboostrs |
|---|---|
| `gamboost(class ~ bbs(x1) + bols(x2), data=d, family=Multinomial())` | `mb.gamboost("class ~ bbs(x1) + bols(x2)", data=d, family=mb.Multinomial())` |

## Families

| R | mboostrs | Notes |
|---|---|---|
| `Gaussian()` | `mb.Gaussian()` | |
| `Binomial()` | `mb.Binomial()` | y in {0,1}; internally coded {-1,+1} |
| `Poisson()` | `mb.Poisson()` | |
| `Laplace()` | `mb.Laplace()` | |
| `GammaReg()` | `mb.GammaReg()` | |
| `GaussReg()` | `mb.GaussReg()` | |
| `AdaExp()` | `mb.AdaExp()` | |
| `Quantile(tau=0.5)` | `mb.Quantile(tau=0.5)` | |
| `Expectile(tau=0.5)` | `mb.Expectile(tau=0.5)` | |
| `Huber()` | `mb.Huber(delta=None)` | With adaptive delta, `cvrisk` starts each fold from its own fit; mboost carries the previous fold's last fit into the next fold's first delta, so its cvrisk depends on the fold order |
| `NBinomial()` | `mb.NBinomial()` | |
| `PropOdds()` | `mb.PropOdds(n_levels=None)` | |
| `CoxPH()` | `mb.CoxPH()` | |
| `Weibull()` | `mb.Weibull()` | |
| `Loglog()` | `mb.Loglog()` | |
| `Multinomial()` | `mb.Multinomial(n_classes=None)` | **gamboost-only.** `glmboost` refuses `family=Multinomial()`, so write a linear multi-class fit as `mb.gamboost("y ~ bols(x)")`. `aic()` and `hatvalues()` raise. `cvrisk` scores held-out rows with the full negative log-likelihood, where mboost leaves the reference class out, so it can choose a later `mstop` |
| `Gehan()` | `mb.Gehan()` | **Offset is 0**, not R's `optimize()` result on a flat function, so `fitted()` differs from R by an additive constant. Gradients, coefficients, `selected` and risk match exactly |
| `Cindex()` | `mb.Cindex()` | |
| `AUC()` | `mb.AUC()` | takes case weights, which R's `AUC()` gradient gets wrong (it returns a vector of length `sum(w)`, not `n`); a weighted fit here has no R counterpart to match |

## Post-fit methods

| R | mboostrs | Notes |
|---|---|---|
| `coef(fit)` | `fit.coef()` | Returns dict |
| `predict(fit, newdata=d)` | `fit.predict(newdata=d)` | Keyword-only args |
| `predict(fit, type="response")` | `fit.predict(type="response")` | |
| `fitted(fit)` | `fit.fitted_` | Attribute; trailing underscore avoids Python builtin |
| `residuals(fit)` | `fit.residuals()` | |
| `logLik(fit)` | `fit.logLik()` | R returns `-risk`; for a Gaussian fit this is the residual sum of squares, not a log-likelihood. `fit.logLik()` plugs in $\hat\sigma^2 = \mathrm{RSS}/n$ and returns a Gaussian log-likelihood with the normalizing constants included. Use `fit.negative_risk()` to reproduce R's value. |
| `AIC(fit)` | `fit.aic()` | Returns AicResult; `.aic` array, `.mstop` int |
| `extract(fit, "coef")` | `fit.extract("coef")` | |
| `extract(fit, "df")` | `fit.extract("df")` | the effective degrees of freedom R reports, `tr(2H - HᵀH)` for the hat matrix `H`, for a `df=` and a `lambda=` learner alike; `None` for `btree`, as in R |
| `extract(fit, "lambda")` | `fit.extract("lambda")` | For spline and block learners, the prepared learner stores the assembled penalty `λK`, so `extract("lambda")` cannot recover `λ` separately. R returns it. Use `extract("df")` to read the effective degrees of freedom. |
| `varimp(fit)` | `fit.varimp()` | `percent=True` by default |
| `confint(fit, level=0.95)` | `fit.confint(level=0.95)` | `B`, `B_mstop`, `seed`, `n_jobs` |
| `mgcv::gam(y ~ s(x, bs="ps", k=24, m=c(2,2)), method="REML")` after boosting (mboost has no refit) | `mb.gam_refit(fit)`; in the mboostrs R package `gam_refit(fit)` | REML refit of the selected terms on their own bases, with `effects()`, `predict()` and `vcov(type="corrected")`; see [](refit.ipynb) |
| `update(fit, mstop=100)` | `fit.update(mstop=100)` | |
| `summary(fit)` | `fit.summary()` | Returns dict |
| `fit[mstop]` | `fit[:mstop]` | Python slice syntax |

### CV and model selection

| R | mboostrs | Notes |
|---|---|---|
| `cvrisk(fit, folds=cv(K=5))` | `fit.cvrisk(type="kfold", B=5)` | `type` is "bootstrap", "subsampling", or "kfold". For `kfold`, `B` is the number of folds (R's `K`). |
| `cvrisk(fit, folds=cv(model=boot, B=25))` | `fit.cvrisk(type="bootstrap", B=25)` | |
| `mstop(cvr)` | `cvr.optimal_mstop` | Returns optimal iteration |
| `fit[mstop(cvr)]` | `fit[:cvr.optimal_mstop]` | |

### Stability selection

| R | mboostrs | Notes |
|---|---|---|
| `stabsel(fit, q=15, PFER=1)` | `mb.stabsel(fit, q=15, PFER=1)` | Returns `StabSelResult` |
| `stabsel(fit, cutoff=0.6, PFER=1)` | `mb.stabsel(fit, cutoff=0.6, PFER=1)` | |

## Plotting

mboostrs has dedicated plot functions. All return an Altair chart for
Vega-Lite rendering. Compose multiple charts with `alt.hconcat`/`alt.vconcat`.

| R `mboost` | mboostrs | Notes |
|---|---|---|
| `plot(fit, which="risk")` | `fit.plot_risk()` | |
| `plot(fit, which="coef")` | `glm.plot_coef_path()` or `glm.plot()` | GLM only |
| `plot(fit, which="partial")` | `fit.plot_partial_effects()` or `fit.plot()` | GAM only |
| `plot(cvr)` | `mb.plot.plot_cvrisk(cvr)` | Fold traces + optimal line |
| `plot(aic_obj)` | `mb.plot.plot_aic(aic_obj)` | AIC path + optimal line |
| `plot(varimp(fit))` | `fit.plot_varimp()` | |
| (no R equivalent) | `fit.plot_selected()` | Selection frequency bars |
| (no R equivalent) | `fit.plot_residuals()` | Residuals vs fitted |
| (no R equivalent) | `mb.plot.plot_confint(fit)` | CI bands (GAM) / forest plot (GLM) |
| (no R `mboost` equivalent) | `mb.plot.plot_lss(lss, parameter="mu")` | Per-parameter plots; also in the mboostrs R package |
| (no R `mboost` equivalent) | `mb.plot.plot_lss_risk(lss)` | LSS risk path; also in the mboostrs R package |
| (no R `mboost` equivalent) | `mb.plot.plot_lss_cvrisk(cvr)` | LSS CV risk; also in the mboostrs R package |
| (no R equivalent) | `mb.plot.plot_multinomial(fit)` | Per-class probability curves |

Subplot composition:

```python
import altair as alt

chart = alt.hconcat(fit.plot_risk(), fit.plot_varimp())
chart.save("diagnostics.pdf")
```

## Differences from R

### No S3 dispatch

R uses `plot()`, `coef()`, `predict()` as generic functions. mboostrs uses
**methods on the model object** (`fit.plot_risk()`, `fit.coef()`,
`fit.predict()`). Functions that take non-model objects (`CvRiskResult`,
`AicResult`) remain module-level (`mb.plot.plot_cvrisk(cvr)`,
`mb.plot.plot_aic(aic)`).

### Formula syntax

mboostrs formulas are plain strings, not R formula objects. The parser handles
`+`, term removal with `-`, whole-frame expansion with `.`, the learner calls
(`bols()`, `bbs()`, `bns()`, `bmono()`, `bconstrained()`, `btree()`,
`brandom()`, `bmrf()`, `bspatial()`, `bkernel()`, `buser()`), the `%X%`
interaction operator, and inline transforms.

Supported in the formula:

- `log(x)`, `scale(x)` and the other inline transforms. The transform recipe is
  stored with the model, so new-data prediction replays it with the training
  parameters.
- Interactions, through `%X%`: `bbs(age) %X% brandom(region)` fits a smooth curve
  whose shape varies by region. See [](../methodology/base_learners.ipynb) for
  the tensor design and Kronecker-sum penalty.

- `I()` for arithmetic over data columns, such as `I(x^2)`; see
  [](#inline-transforms-identity).

Not supported:

- `x1:x2` as R spells it — use `%X%` for a tensor interaction, or
  `bols(x1, by=x2)` for a varying coefficient.

#### Fitting without an intercept

`glmboost()` takes `- 1`, `+ 0` and a leading `0 +` as R does: the intercept
learner leaves the candidate set and the boosting offset stays, so "no
intercept" means no intercept *learner*, not a fit through the origin. The
covariates are still centered (`center=True`), and the fit warns `model with
centered covariates does not contain intercept`, as mboost's does.

```python
# R:        glmboost(y ~ x1 + x2 + x3 - 1, data = d)
fit = mb.glmboost("y ~ x1 + x2 + x3 - 1", data=d)
```

Fitted values, predictions, coefficients and `cvrisk` match mboost 2.9.11 to
within 4e-15 for Gaussian and Binomial fits, in both frontends. The other
fitting functions refuse `- 1`: `gamboost` formulas name base-learners, as in
mboost, and an LSS model always carries an intercept per parameter.

`gamboost("y ~ bols(x1) + bols(x2)")` is not a substitute. Each `bols` term
carries its own constant column, which reproduces the no-intercept fit for a
Gaussian loss only (to 4e-15); for Binomial it differs by 0.011. Neither is
`bols(x, intercept=False)`, which removes the constant from inside one term's
design and changes the fit.

### Data

mboostrs accepts a dict of NumPy arrays or a pandas DataFrame. Column names
in the dict become the names used in formulas and coefficient labels.

### Features in R without an mboostrs equivalent

The distribution catalogue is smaller than `gamlss.dist`, which provides
about 116 families through `gamboostLSS::as.families()`. `mboost::Family()` can
build a family from R closures at run time; mboostrs provides 27 compiled,
hand-written distributional families and requires new family implementations
in Rust. The densities and gradients of the included families agree with
`gamlss.dist`.

Smaller gaps:

- Single-parameter families as *objects* (`GaussianMu()`, `GaussianSigma()`).
  The capability exists — see
  [](gamboostlss.ipynb#fitting-one-parameter-with-the-others-fixed) — it is spelled with
  per-parameter `mstop` rather than with a family per parameter.
- `blackboost` / `blackboostLSS` — tree-based boosting as its own entry point.
  `btree()` inside `gamboost` covers the same learners.
- `GaussClass` and `RCG` families; `FP` (fractional polynomials). (`Hurdle`
  is implemented, as `mb.Hurdle`.)
- `ACD` residuals diagnostic (`which="acfd"`)
- `nup` negative-gradient-vs-predictor diagnostic
- Formula notation `:` and `*` for interactions. `I()` evaluates arithmetic
  over data columns, not arbitrary R code.
- On-line stopping (early stopping within `fit()`)
- `$`, `[[` accessors for model components
- `downstream.test` post-selection inference

In R mboost, `bss` and `bns` raise an error that points to `bbs`.
`brad` is implemented (see
[](base_learners.ipynb#radial-basis-surfaces-brad)), though it selects its
centres differently from `fields::cover.design`, so fits are comparable rather
than identical.

## Common pitfalls

### String formulas

R accepts `y ~ x` as an expression. mboostrs requires a **string**: `"y ~ x"`.
Omitting the quotes produces `SyntaxError`.

### Family names

Family names match R exactly; only the `mb.` namespace prefix differs:

| R | mboostrs |
|---|---|
| `GammaReg()` | `mb.GammaReg()` |
| `GaussReg()` | `mb.GaussReg()` |
| `Quantile()` | `mb.Quantile()` |
| `Expectile()` | `mb.Expectile()` |
| `AdaExp()` | `mb.AdaExp()` |
| `NBinomial()` | `mb.NBinomial()` |
| `Weibull()` (survival) | `mb.Weibull()` |

If `GammaReg()` fails to find a Python class, check the capitalization —
class names are case-sensitive.

(r-migration-indexing)=
### 0-indexed vs 1-indexed

- `brandom` group codes are 1-based in R. mboostrs accepts both 0-based and
  1-based integer codes: if the smallest code is 0 the codes are read as 0-based,
  otherwise as 1-based. Codes outside the resulting range raise `ValueError`.
- `selected` arrays are 0-based.
- Array slicing: `fit[:mstop]` (not `fit[1:mstop]`).

### Binomial response coding

Use $y \in \{0, 1\}$; the Rust engine re-codes to $\{-1, +1\}$ internally.
A response already coded $\{-1, +1\}$ gives a bit-identical fit — same offset,
same coefficients, same risk — because that re-coding is idempotent: it
recognises a $-1$ and leaves it alone. Nothing needs undoing, and a ported R
script that already uses $\pm 1$ can be left as it is.

### AIC for survival families

Calling `fit.aic()` on a `CoxPH`, `Loglog`, `Lognormal`, or `Weibull` model
raises `NotImplementedError`. The degrees of freedom that `aic()` uses are the
trace of the hat matrix for squared-error loss, which is not the effective
dimension of a censored or partial likelihood. Choose `mstop` with `cvrisk`
for these families.

### `coef()` keys

R's `coef()` returns a named numeric vector whose names are the *learner*
spellings: `coef(gamboost(y ~ bbs(x)))` is named `bbs(x)`. mboostrs returns a
`dict[str, np.ndarray]` keyed by the **variable**, so the same fit gives
`fit.coef()["x"]`, and `fit.coef()["bbs(x)"]` raises `KeyError`.

`fit.base_learner_names()` lists the keys for a fit in hand. A `gamboost` fit
also carries `"(offset)"`; a `glmboost` fit carries `"(Intercept)"` instead,
because there the intercept is a fitted learner rather than the boosting
starting value.

### Weights length mismatch

Specifying `weights` with fewer/more elements than `data` rows triggers a
Python `ValueError` from the model constructor with the expected and actual lengths.

### Pickling

Fitted models pickle and unpickle with the standard `pickle` module, and an
unpickled model predicts. Load a pickle with the same mboostrs version that
wrote it; for a file that other versions or the R and Rust readers can load,
use the JSON model artifact (`fit.save(path)` and `mb.load_model(path)`).

### The `bmrf` neighbourhood matrix

The formula parser recognizes `bmrf(region)` but cannot read a matrix out of a
formula string, so `K` has to be bound by name through `env=`:

```python
mb.gamboost("y ~ bmrf(region, K=K_mrf, df=4)", data=data,
            family=mb.Gaussian(), env={"K_mrf": K})
```

Construction and fitting happen in one call to `gamboost()`. `bconstrained`'s constraint matrix `C` is supplied the
same way.

### `mstop` indexing: off-by-one

`risk_[i]` is the risk *after* $i$ iterations. `aic[mstop - 1]` is the AIC
at the optimal iteration. The cvrisk grid starts at 0 (offset-only), but
`optimal_mstop` never returns 0 — R's `mstop(cvrisk_object)` always
selects at least one boosting iteration. Slicing with `fit[:mstop]`
includes `mstop` iterations (Python exclusive upper bound).

## New capabilities not in R mboost

These mboostrs features have no direct R `mboost` equivalent:

| Feature | mboostrs API |
|---|---|
| Diagnostic plots | `plot_varimp`, `plot_selected`, `plot_residuals`, `plot_confint`, `plot_lss`, `plot_multinomial` |
| Parallel bootstrap | `fit.confint(..., n_jobs=...)` |
| scikit-learn estimators | `mboostrs.sklearn` |
| Pickle serialization of fitted models | `pickle.dumps(fit)` |
| Non-cyclic LSS fitting | `mb.gamboostLSS(..., method="noncyclic")` |
| Compositional response family | `mb.DirichletLSS` |
| Portable, schema-versioned JSON model artifact (Python, R and Rust readers; distributional models in Python) | `fit.save(path)`, `mb.load_model(path)` |
| Conformal prediction intervals (split, CV+ and conformalized quantile regression) | see [](../methodology/conformal.ipynb) |
| Prediction contribution decomposition | `model.predict(which="x1", aggregate="cumsum")` |
| Structured summary (JSON) | `model.summary(as_dict=True)` |
| Extraction methods | `model.variable_names()`, `model.model_frame()`, `model.model_matrix()` |
| Model comparison table | `mb.gaic_table(model1, model2)` |
| Multi-metric comparison | `mb.compare_models(model1, model2, metrics=("risk", "GAIC"))` |
| LSS family scanning | `mb.scan_lss_families(formula, data, families=[...])` |
| mstop profiling | `mb.profile_mstop(model, grid=[...], metric="risk")` |
| Weighted LSS fits | `mb.glmboostLSS(..., weights=w)` |
| LSS GAIC | `lss_model.gaic(k=2)` |
| R diff harness | `mb.compare_to_r(CompareSpec(...))` |
| Standalone diagnostics | `mb.quantile_residuals(model)`, `mb.pit(model)`, `mb.q_stats(model)` |

## See also

- [](../appendix/r_parity_matrix) — feature-by-feature comparison of R and mboostrs support.
- [](quickstart.ipynb) — a quick end-to-end example to get started.